In [ ]:
from pathlib import Path
from IPython.display import Image, display

cover_image_path = Path("/kaggle/input/datasets/pilkwang/pilkwang-public-dataset-for-notebooks-figures/biohub_cover.png")
if cover_image_path.exists():
    display(Image(filename=str(cover_image_path)))

## Closing Note for This Model Line

Thank you for all the interest, upvotes, forks, and thoughtful tweaks along the way. They have helped me a lot, and I hope this notebook has been useful for others as well.

This notebook is built around one learned tracking pipeline:

- `TemporalUNet3D` for cell-center detection.
- Node cross-attention transformer for temporal edge scoring.
- ILP graph construction for selecting the tracking graph.
- Deterministic graph repair for motion relinking, bounded gaps, conservative divisions, and line-fit smoothing.

I plan to wrap up this branch after this checkpoint cycle and move on to the next modeling stage, so I do not expect major updates to this notebook after that point. I would be happy to keep exchanging ideas in the next notebook, or in other competitions where our paths cross again.

# Biohub Cell Tracking: Recall With Cleaner Repair
The learned TemporalUNet3D + node-transformer + ILP tracker with a high-recall detection threshold and tighter gap/division repair caps.

Pipeline summary:

- **Center model:** `TemporalUNet3D` predicts cell-center logits from short 3D time windows.
- **Edge model:** a node cross-attention transformer scores candidate parent-child links between adjacent frames.
- **Graph optimizer:** an ILP selects a sparse tracking graph from learned edge scores and event costs.
- **Deterministic repair:** physical-motion relinking, bounded gap repair, conservative division recovery, and line-fit coordinate smoothing.

This variant keeps a high-recall detection threshold and uses tighter gap/division repair caps.

## Current Model and Submission Logic

The submitted graph is produced by a single learned tracker artifact followed by geometry-constrained graph repair.

**Learned / optimization components**

- `TemporalUNet3D` center detector: predicts cell-center logits from short 3D time windows.
- Node cross-attention edge predictor: scores candidate parent-child links between adjacent frames.
- ILP graph constructor: selects a sparse tracking graph from learned edge scores and event costs.

**Deterministic post-processing**

- Physical-motion relinking.
- Bounded one-frame and two-frame gap repair.
- Conservative division recovery.
- Topology-preserving line-fit coordinate smoothing.

This profile keeps a high-recall detection threshold, then reduces synthetic gap additions, gap2 recovery, and conservative division recovery to test whether the remaining loss is over-repair.

### Training Objective

For a short temporal window, the model encodes the downsampled 3D volumes with a
`TemporalUNet3D` encoder. The encoder output at frame $t$ is converted into a
center logit field by a $1\times1\times1$ convolution.

$$
F_t = h_\theta(V_{t:t+W-1})_t,
\qquad
 a_t(\mathbf r) = q_\theta(F_t(\mathbf r)),
\qquad
p_t(\mathbf r)=\sigma(a_t(\mathbf r)).
$$

Ground-truth node voxels are positives and all other voxels are weak negatives.
For one frame, the detection target is $y_t(\mathbf r)\in\{0,1\}$. The per-sample
weights normalize positive and negative mass separately:

$$
w_+(b)=\frac{1}{N_+(b)},
\qquad
w_-(b)=\frac{\alpha}{N_-(b)},
\qquad \alpha=0.01.
$$

The detection loss is weighted binary cross entropy on logits:

$$
\mathcal L_{\mathrm{det}}
=
\frac{1}{B}\sum_{b=1}^{B}\sum_{\mathbf r}
 w_b(\mathbf r)
\,\mathrm{BCELogit}(a_b(\mathbf r),y_b(\mathbf r)).
$$

Local maxima of $p_t$ become candidate detections. For consecutive frames, node
features are sampled from the UNet feature maps and passed through a bidirectional
cross-attention transformer. The edge logit from source node $i$ to target node
$j$ is

$$
\ell_{ij}=g_\phi\left([F_t(\mathbf r_i),\psi(\mathbf r_i,t)],
[F_{t+1}(\mathbf r_j),\psi(\mathbf r_j,t+1)],
\mathbf r_j-\mathbf r_i\right).
$$

During training, edge probabilities are normalized over possible parents of each
target:

$$
P_{ij}=\mathrm{softmax}_{i}(\ell_{ij}).
$$

This permits one parent to connect to two daughters while discouraging many
parents for the same target. Because annotations are sparse, the edge loss is
computed only on rows or columns that touch an annotated edge:

$$
\mathcal M_{ij}=1
\quad\mathrm{if}\quad
\sum_j y_{ij}>0\quad\mathrm{or}\quad\sum_i y_{ij}>0.
$$

The edge objective is focal-style binary cross entropy:

$$
\mathcal L_{\mathrm{edge}}
=
\mathrm{mean}_{(i,j):\mathcal M_{ij}=1}
(1-P^{*}_{ij})^2\,\mathrm{BCE}(P_{ij},y_{ij}),
$$

where $P^{*}_{ij}=P_{ij}$ when $y_{ij}=1$ and $P^{*}_{ij}=1-P_{ij}$ otherwise.
The trained checkpoint minimizes

$$
\mathcal L
=
\mathcal L_{\mathrm{edge}}
+\lambda_{\mathrm{det}}\mathcal L_{\mathrm{det}},
\qquad \lambda_{\mathrm{det}}=1.
$$

### Submission Graph Construction

At inference time, detections are local maxima above the fixed probability
threshold $\tau=0.985$. Candidate links come from the learned edge predictor and
are solved with the ILP settings in `RUN_CONFIG`.

All geometry after prediction is measured in microns:

$$
d_{\mu\mathrm{m}}(i,j)=
\sqrt{(1.625\Delta z)^2+(0.40625\Delta y)^2+(0.40625\Delta x)^2}.
$$

The motion relinker rebuilds one-step temporal links from detected nodes. If a
node has a predecessor, its next position is extrapolated by

$$
\hat{\mathbf r}_{i,t+1}=\mathbf r_{i,t}
+\lambda_v(\mathbf r_{i,t}-\mathbf r_{i,t-1}).
$$

A Hungarian assignment is then solved with cost

$$
C_{ij}=d_{\mathrm{motion}}(i,j)+0.05d_{\mathrm{raw}}(i,j)-\beta P_{ij}^{\mathrm{learned}}.
$$

This notebook uses

$$
\lambda_v=0.52,\qquad
\beta=0.78,\qquad
R_{\mathrm{tight}}=6.2\,\mu\mathrm{m},\qquad
R_{\mathrm{relaxed}}=10.4\,\mu\mathrm{m}.
$$

One-frame gap repair can insert or reuse a middle-frame node when a track end at
$t$ and a track start at $t+2$ are physically close enough:

$$
d_{\mu\mathrm{m}}(i,j)\le 2g,
\qquad g=5.9\,\mu\mathrm{m}.
$$

Two-missing-frame recovery is controlled by a much smaller cap. When enabled,
it requires

$$
d_{\mu\mathrm{m}}(i,j)\le R_2,
\qquad
\frac{d_{\mu\mathrm{m}}(i,j)}{3}\le s_2,
$$

with

$$
R_2=9.7\,\mu\mathrm{m},
\qquad
s_2=4.05\,\mu\mathrm{m},
\qquad
\rho_2=0.0032.
$$

Finally, line-fit smoothing modifies only coordinates, not graph topology. For a
linear local track neighbourhood,

$$
\mathbf r'_i=(1-w)\mathbf r_i+w\,\mathrm{LineFit}_{\mathcal N(i)}(t_i),
\qquad w=0.74.
$$

The submission writer emits only the required node and edge rows and validates
that every hidden-test dataset appears in `submission.csv`.

# Cell description:
Here's what it does:

This is the notebook's central configuration cell. It doesn't do any computation itself — it just sets up paths, imports, and every tunable knob used later in the pipeline. Breaking it down:

1. Imports (lines 219–234)
Standard library utilities the rest of the notebook will need: csv/json for I/O, importlib.util for dynamically loading modules, shutil/tempfile/zipfile for handling the artifact package, subprocess for shelling out (likely pip installs), pathlib.Path for paths, and pandas for the submission dataframe. from __future__ import annotations just makes type hints lazily-evaluated strings (lets it use modern type-hint syntax regardless of Python version).

2. Data & working-directory paths (lines 236–251)
Locates the competition's test data — it tries two possible Kaggle input locations (COMP_DIR_CANDIDATES) and picks whichever exists, falling back to the first if neither does. TEST_DIR can be overridden via the BIOHUB_TEST_DIR env var (useful for local testing off-Kaggle). WORKING_DIR similarly detects whether it's running on Kaggle (/kaggle/working) or locally (.), and defines where the repo, submission.csv, and run_stats.csv will be written.

3. Model/artifact config (lines 254–263)
Names the method (unet_transformer), the path to trained weights, and where to find the "support artifact" (a bundled package of code + weights + optionally offline wheels) via TARGET_ARTIFACT_SLUG and PRIMARY_ARTIFACT_MANIFEST. ALLOW_ARTIFACT_FALLBACK controls whether an older/mismatched artifact is accepted.

4. Inference hyperparameters (lines 266–331)
All overridable via BIOHUB_* environment variables, each falling back to a hardcoded default — this is the pattern used throughout:

Detection threshold, batch size, ILP (integer linear programming) on/off and its edge/appearance/disappearance/division cost weights.
Post-processing knobs for the deterministic "graph repair" step mentioned in the earlier markdown cells: max edge distance, motion-based relinking distances/weights, division geometry filters, gap-closing (1-frame and 2-frame gap recovery) distance/fraction caps, short-track filtering, and line-fit smoothing weight/window.
This lets someone re-run the notebook with different repair aggressiveness just by setting environment variables, without editing code.

5. Config display and sanity print (lines 333–400)
CONFIG_DISPLAY collects every one of those settings into a dict, then the cell prints whether COMP_DIR/TEST_DIR actually exist (a quick sanity check that paths resolved correctly) and dumps the full config as pretty JSON — so the top of the notebook's output log documents exactly what settings produced that run's submission.

In short: it's a parameter/config block, not an algorithmic one — everything computationally interesting (the U-Net, transformer, ILP, repair heuristics) happens in cells after it, driven by these constants.

In [1]:
from __future__ import annotations

import csv
import importlib.util
import json
import math
import os
import shutil
import subprocess
import tempfile
import zipfile
import sys
import time
from pathlib import Path

import pandas as pd

# ===== Setting paths to the competition data and model ==============
COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [
    Path(f"/kaggle/input/competitions/{COMPETITION}"),
    Path(f"/kaggle/input/{COMPETITION}"),
]
COMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])
_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()
TEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test" #Location of test data
# =====================================================================
# ============= Defining working directory (own directory) ============
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
SUBMISSION_PATH = WORKING_DIR / "submission.csv"
RUN_STATS_PATH = WORKING_DIR / "run_stats.csv"
# =====================================================================


METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth" #location of weights for trained model
EXPERIMENT_TAG = "candidate_20_200ep_recall_clean_repair"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")#get that environment variable
    #using the key "BIOHUB..." if it is not defined, the variable is defined to "biohub-tracking-support-pack-50ep-v1"
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",
)) #same idea as before, but now it is defined as a Path
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"

# ===== Variable definitions that are inputs when making predictions with the baseline model ====================
DET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.985")) #Threshold to accept a voxel as node
UNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4")) #batch size of videos
USE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0" #In this case the condition is 'True', so the ILP is used
ILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0")) #weight in the cost function for the edges (linking)
ILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1")) #Weights if cells suddenly appear 
ILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1")) #Weights if cells suddenly disappear
ILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0")) #weight for division cells

# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.
SLICE = os.environ.get("BIOHUB_SLICE", "").strip()

# If dependencies are not already installed and no offline wheels are attached,
# this controls whether the notebook attempts PyPI installation.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"

# Output-level graph post-processing.
OUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.2"))
OUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"
OUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"
OUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"
OUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"
OUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"
MOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "5.95"))
MOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))
MOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.50"))
MOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.76"))
MOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "3200"))

OUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"
DIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))
DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))
DIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"
OUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"
GAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))
GAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "5.9"))
GAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"
GAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.1"))
GAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.045"))
GAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "1900"))
GAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"
GAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))
GAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))
GAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))

OUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "0") != "0"
OUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "4"))
OUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"

OUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"
OUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.74"))
OUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))

OUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "1") != "0"
GAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "9.5"))
GAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.0"))
GAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0026"))
GAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "120"))
GAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"
GAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.0040"))

OUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"
SAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.6"))
SAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "6.9"))
SAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.4"))
SAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))
SAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.0036"))

CONFIG_DISPLAY = {
    "experiment_tag": EXPERIMENT_TAG,
    "method": METHOD,
    "weights": WEIGHTS_RELATIVE,
    "target_artifact_slug": TARGET_ARTIFACT_SLUG,
    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),
    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,
    "det_threshold": DET_THRESHOLD,
    "unet_batch_size": UNET_BATCH_SIZE,
    "use_ilp": USE_ILP,
    "ilp_edge_weight": ILP_EDGE_WEIGHT,
    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,
    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,
    "ilp_division_weight": ILP_DIVISION_WEIGHT,
    "slice": SLICE,
    "allow_pip_install": ALLOW_PIP_INSTALL,
    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,
    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,
    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,
    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,
    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,
    "output_motion_relink": OUTPUT_MOTION_RELINK,
    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,
    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,
    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,
    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,
    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,
    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,
    "div_parent_max_um": DIV_PARENT_MAX_UM,
    "div_sister_max_um": DIV_SISTER_MAX_UM,
    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,
    "output_gap_close": OUTPUT_GAP_CLOSE,
    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,
    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),
    "gap_close_um": GAP_CLOSE_UM,
    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,
    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,
    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,
    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,
    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,
    "gap_refine_win_z": GAP_REFINE_WIN_Z,
    "gap_refine_win_yx": GAP_REFINE_WIN_YX,
    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,
    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,
    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,
    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,
    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,
    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,
    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,
    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,
    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,
    "gap2_max_step_um": GAP2_MAX_STEP_UM,
    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,
    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,
    "gap2_require_context": GAP2_REQUIRE_CONTEXT,
    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,
    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,
    "safe_div_max_um": SAFE_DIV_MAX_UM,
    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,
    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,
    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,
    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,
}

print("Biohub learned UNet + node-transformer + ILP submission")
print("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())
print("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())
print(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))

Biohub learned UNet + node-transformer + ILP submission
COMP_DIR: /kaggle/input/competitions/biohub-cell-tracking-during-development exists: True
TEST_DIR: /kaggle/input/competitions/biohub-cell-tracking-during-development/test exists: True
{
  "allow_artifact_fallback": false,
  "allow_pip_install": false,
  "det_threshold": 0.985,
  "div_drop_to_single_if_bad": true,
  "div_parent_max_um": 10.5,
  "div_sister_max_um": 8.0,
  "experiment_tag": "candidate_20_200ep_recall_clean_repair",
  "gap2_frame_frac_cap": 0.004,
  "gap2_max_links_abs": 120,
  "gap2_max_links_frac": 0.0026,
  "gap2_max_step_um": 4.0,
  "gap2_max_total_um": 9.5,
  "gap2_require_context": true,
  "gap_close_effective_max_gap": 1,
  "gap_close_max_added_abs": 1900,
  "gap_close_max_added_frac": 0.045,
  "gap_close_max_gap": 1,
  "gap_close_reuse_existing": true,
  "gap_close_reuse_um": 3.1,
  "gap_close_um": 5.9,
  "gap_refine_max_shift_um": 3.2,
  "gap_refine_synthetic": true,
  "gap_refine_win_yx": 3,
  "gap_refine_

## Artifact and Dependency Setup

The notebook expects a compact support artifact containing the inference source,
trained weights, and optionally offline dependency wheels. The primary Kaggle
attachment path is:

```text
/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json
```

The setup cell first uses already-installed modules, then attached wheels, and
only attempts an internet install when explicitly enabled for local development.
By default, the artifact resolver accepts only the 50-epoch package named by
`TARGET_ARTIFACT_SLUG`; set `BIOHUB_ALLOW_ARTIFACT_FALLBACK=1` only for local
debugging against an older package.


##### The following block deals with all paths and the associated files. It looks for different files trhough the whole input directory. Additionally, it installs required packages dealing with incompatibility errors 

In [2]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]

# The safe path for offline reruns is to use attached wheels.
# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


ARTIFACTS: /kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1
Has offline wheels: True
Artifact name: biohub-tracking-support-pack-400ep-snapshot-v1
Weight sha256: 12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771
Weight path: weights/unet_transformer/split_0/edge_predictor_best.pth
Installing missing packages from offline package dirs: ['polars']
Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.
Offline package dirs: ['/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/wheels']
Offline dependency install succeeded.
Installing missing packages from offline package dirs: ['tracksdata', 'zarr', 'pyscipopt', 'geff', 'geff_spec', 'ilpy', 'imagecodecs', 'rustworkx', 'numcodecs', 'donfig', 'bidict']
Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.
Offline package dirs: ['/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-

/usr/local/lib/python3.12/dist-packages/dask/array/image.py:7: FutureWarning: `find_available_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  from skimage.io import imread as sk_imread
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `reset_plugins` is deprecated since version 0.25 and will be removed in version 0.27. The plugin infrastructure of `skimage.io` is deprecated. Instead, use `imageio` or other I/O packages directly.
  return _bootstrap._gcd_import(name[level:], package, level)


Required graph/Zarr/ILP packages import successfully.
Inference repo: /kaggle/working/tracking_repo
Weights: /kaggle/working/tracking_repo/weights/unet_transformer/split_0/edge_predictor_best.pth


## Predict Candidate Graphs

The inference step writes one `.geff` graph per test video. Keeping graph
prediction separate from CSV conversion makes the graph repair and diagnostics
transparent.


# Cell description:
This cell executes the ```predict_unet_transformer.py``` from the baseline model without any modification

In [3]:
def list_test_stems() -> list[str]:
    if not TEST_DIR.exists():
        raise FileNotFoundError(f"Test directory does not exist: {TEST_DIR}")
    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith(".zarr"))
    if not stems:
        raise FileNotFoundError(f"No test .zarr files found in {TEST_DIR}")
    return stems


test_stems = list_test_stems()
print(f"Found {len(test_stems)} test videos")
print(test_stems[:10])

splits_path = REPO_DIR / "kaggle_test_splits_50ep.json"
splits_path.parent.mkdir(parents=True, exist_ok=True)
splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}], indent=2))

predict_cmd = [
    sys.executable,
    "scripts/predict_unet_transformer.py",
    "--data-dir",
    str(TEST_DIR),
    "--splits",
    str(splits_path.name),
    "--split",
    "0",
    "--weights",
    WEIGHTS_RELATIVE,
    "--unet-batch-size",
    str(UNET_BATCH_SIZE),
    "--det-threshold",
    str(DET_THRESHOLD),
    "--ilp-edge-weight",
    str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight",
    str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight",
    str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight",
    str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    predict_cmd.append("--use-ilp")
if SLICE:
    predict_cmd.extend(["--slice", SLICE])

start_time = time.time()
print(" ".join(predict_cmd))
# subprocess.run(predict_cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)
# predict_seconds = time.time() - start_time
print(f"Prediction completed in {predict_seconds / 60:.2f} minutes")


Found 4 test videos
['44b6_0113de3b', '44b6_0b24845f', '6bba_05b6850b', '6bba_05db0fb1']
/usr/bin/python3 scripts/predict_unet_transformer.py --data-dir /kaggle/input/competitions/biohub-cell-tracking-during-development/test --splits kaggle_test_splits_50ep.json --split 0 --weights weights/unet_transformer/split_0/edge_predictor_best.pth --unet-batch-size 4 --det-threshold 0.985 --ilp-edge-weight -1.0 --ilp-appearance-weight 0.1 --ilp-disappearance-weight 0.1 --ilp-division-weight 1.0 --use-ilp


NameError: name 'predict_seconds' is not defined

## Build `submission.csv`

Rows are streamed directly to disk with the required schema. This avoids holding
the full hidden-test submission table in memory.

The standard gap closer intentionally handles only one missing frame. Two-missing-frame repair is handled by the stricter `gap2` pass so that a loose environment override cannot introduce non-consecutive edges.


In [ ]:
import tracksdata as td
import numpy as np
import blosc2
from scipy.optimize import linear_sum_assignment

SUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
CSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]
VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)


def graph_from_geff(path: Path):
    '''This returns an IndexedRXGraph object representing the entire cell lineage tree for that embryo sample (for eac
    cell there is a complete graph: t,z,y,x) and each .geff file corresponds to one video.
    Example:
    graph.node_attrs().iter_rows(named=True)   # -> rows with keys: node_id, t, z, y, x
    graph.edge_attrs().iter_rows(named=True)   # -> rows with keys: source_id, target_id, edge_prob
    '''
    graph = td.graph.IndexedRXGraph.from_geff(path)
    return graph[0] if isinstance(graph, tuple) else graph


def edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:
    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]
    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]
    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:
    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]
    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]
    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]
    return math.sqrt(dz * dz + dy * dy + dx * dx)


def node_point(node: dict[str, object]) -> tuple[float, float, float]:
    return (float(node["z"]), float(node["y"]), float(node["x"]))


def edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:
    prob = edge.get("edge_prob")
    prob_value = float(prob) if prob is not None else 0.0
    return prob_value, -float(edge["distance_um"])


def _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:
    return max(nodes_by_id) + 1 if nodes_by_id else 1



def read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:
    '''
    oads a single 3D microscopy frame (all z, y, x voxels at one timepoint t) 
    for a given dataset, from a Zarr array stored on disk
    '''
    if t in frame_cache:
        return frame_cache[t]
    zarr_path = TEST_DIR / f"{dataset}.zarr"
    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())
    shape = tuple(int(v) for v in meta["shape"])# the metadata contains t,x,y,z
    dtype = np.dtype(meta["data_type"]) #reads the dtype
    frame_shape = shape[1:] #from shape(read above) drops the t and keeps only x,y,z
    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"
    try:
        #fast and manual reading skipping the use of zar libraries
        raw = chunk_path.read_bytes() #reads the bytes associated to that frame (t)
        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)#decompressed bytes as a flat array
        if arr.size == int(np.prod(frame_shape)): #checks that the decompressed data matches the
            #expected dimension from the metadata
            frame = arr.reshape(frame_shape).copy()#if the size matches, re-shapes to x,y,z and
            #makes a copy as np.frombuffer gives a read only object
            frame_cache[t] = frame #add the frame (x,y,z) coordinates of all nodes at frame t
            #to the "frame_cache[t]"
            return frame
    except Exception:
        pass
    #if the above fails, it reads the frame using the standard (and slow) zarr library
    import zarr
    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])
    frame_cache[t] = frame
    return frame


def refine_synthetic_midpoint(
    dataset: str | None,
    t: int,
    midpoint: tuple[float, float, float],
    frame_cache: dict[int, np.ndarray],
    stats: dict[str, int],
) -> tuple[float, float, float]:
    '''
    Obtains the intensity values around the "mid point" value and uses this intensity values
    as weigths to refine the position of the synthetic node. The idea is that more intese voxels
    would be associated to cells, and the initial "mid point" naive estimation can be improved if
    this is taken into account
    '''
    if not GAP_REFINE_SYNTHETIC or dataset is None:
        return midpoint
    try:
        frame = read_test_frame(dataset, t, frame_cache)
        z, y, x = [int(round(v)) for v in midpoint]
        z0 = max(0, z - GAP_REFINE_WIN_Z)
        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)
        y0 = max(0, y - GAP_REFINE_WIN_YX)
        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)
        x0 = max(0, x - GAP_REFINE_WIN_YX)
        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)
        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)
        if patch.size == 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        baseline = float(np.percentile(patch, 20.0))
        weights = np.maximum(patch - baseline, 0.0)
        total = float(weights.sum())
        if total <= 0:
            stats["gap_refine_failed"] += 1
            return midpoint
        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]
        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]
        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]
        refined = (
            float((weights * zz).sum() / total),
            float((weights * yy).sum() / total),
            float((weights * xx).sum() / total),
        )
        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:
            stats["gap_refine_rejected_shift"] += 1
            return midpoint
        stats["gap_refined_synthetic"] += 1
        return refined #weighted mean of positions (voxel coordinates), weighted by intensity, 
    except Exception:
        stats["gap_refine_failed"] += 1
        return midpoint


def _position_um(node: dict[str, object]) -> np.ndarray:
    return np.array(
        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1],
         float(node["x"]) * VOXEL_SCALE_UM[2]],
        dtype=np.float64,
    )


def motion_relink_edges(
    nodes_by_id: dict[int, dict[str, object]],
    stats: dict[str, int],
    learned_edge_probs: dict[tuple[int, int], float] | None = None,
) -> list[dict[str, object]]:
    '''
    This function computes "ids_by_t" where the key is the time frame and the values are all nodes id
    belonging to that time step. Then it estimates a ditance the source node would move and computes
    the distance between all target and sources nodes, with this information and the probability that
    node i is linked to node j, it creates a custome cost function between all nodes sorce-target
    (O(nxm)) for the Hungarian algorithm. Solves the Hungarian problem and returns a set of new
    source node_id, target node_id with other information used to compute this new relationship
    '''
    if not OUTPUT_MOTION_RELINK or not nodes_by_id:
        return []

    learned_edge_probs = learned_edge_probs or {}

    def learned_prob(source_id: int, target_id: int) -> float:
        '''
        Returns the probability of a link beween source_id and target_id nodes. The probability
        was previously retrieved from the .geff file.
        '''
        value = learned_edge_probs.get((source_id, target_id), 0.0)
        try:
            value = float(value)
        except (TypeError, ValueError):
            return 0.0
        if not np.isfinite(value):
            return 0.0
        if value < 0.0 or value > 1.0:
            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))
        return float(np.clip(value, 0.0, 1.0))

    # == this block groups nodes id by time frame, so all nodes at time t are now belonging to the ==
    # == same key in the dictionary. Example: if nodes_by_id were
    # == {1: {"t": 0, ...}, 2: {"t": 1, ...}, 3: {"t": 0, ...}, 4: {"t": 1, ...}}
    # == the result is:
    # == ids_by_t = {0: [1, 3], 1: [2, 4]} where in key 0  the nodes id 1 and 3 belong to that key

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id)
    for ids in ids_by_t.values():
        ids.sort()

    frame_sizes = [len(ids) for ids in ids_by_t.values()]# tell mes how many nodes are at each time t
    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES: #if there are too many
        #nodes on one frame, assigns the variable "motion_relink_skipped_large_frame" to 1 
        #later this will be used to skipt the Hungarian assigment algorithm because it will be too 
        #expensive O(nxm)
        stats["motion_relink_skipped_large_frame"] = 1
        return []

    # creates a new dictionary with key node ide and values the x,y,z coordiantes in physical units (um)
    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}
    predecessor_position_um: dict[int, np.ndarray] = {}
    selected_edges: list[dict[str, object]] = []

    def assign_pass(
        source_ids: list[int],
        target_ids: list[int],
        gate_um: float,
    ) -> list[tuple[int, int, float, float, float]]:
        if not source_ids or not target_ids:
            return []
        big = gate_um * 1000.0 + 1.0
        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)
        raw_dist = np.full_like(cost, np.inf)
        motion_dist = np.full_like(cost, np.inf)
        prob_matrix = np.zeros_like(cost)
        for i, source_id in enumerate(source_ids):
            source_pos = position_um[source_id] #obtains the physical positions (x,y,z) for each node_id
            prev_pos = predecessor_position_um.get(source_id)#predecessor_... key is the node_id
                #and value is the position, the key of the target_id and the position of the 
                #previous time step node associated to that target (source node)
            if prev_pos is None:#this happens for the first time step
                predicted = source_pos
            else:
                #IMPORTANT: this predicts or estimates the next position (coordinate) that the source
                #node would have
                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)
            for j, target_id in enumerate(target_ids):
                target_pos = position_um[target_id]
                raw = float(np.linalg.norm(target_pos - source_pos)) #computes the physical
                #euclidian distance between target and source nodes. One source node and loops trhough
                #all nodes (target nodes) in the next time step. Independently if they are linked
                #or not.
                if raw > gate_um: #for source and target nodes that are far away for more than 
                            #gate_um distance, it doesn't do anything
                    continue
                motion = float(np.linalg.norm(target_pos - predicted))#now computes the 
                #physical euclidian distance between the target note and what it would be predicted.
                prob = learned_prob(source_id, target_id)
                #i:source, j:target
                raw_dist[i, j] = raw #distace target,source
                motion_dist[i, j] = motion #ditace target, estiamted target
                prob_matrix[i, j] = prob #probability of link between node 1 and j
                #cost function: Important as this is used for the Hungarian algorithm.
                #larger distance between nodes -> higher cost
                #higher probability of a connection -> lower cost
                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob
        row_ind, col_ind = linear_sum_assignment(cost)#this solves the assigment problem: source->target
        #pair (i,j) that minimizes the cost (Hungarian algorithm)
        matches: list[tuple[int, int, float, float, float]] = []
        for r, c in zip(row_ind, col_ind):
            if cost[r, c] >= big: #if the cost associated to the matching source-target node is larger
                #than big, it doesn't do anything
                continue
            #if the cost function is smaller than "big" it creates a new list with the source-target
            #relationship and the distance, espected distance and probability of that link (obtained,
            #with the Temporal-UNet)
            matches.append((
                source_ids[int(r)],
                target_ids[int(c)],
                float(raw_dist[r, c]),
                float(motion_dist[r, c]),
                float(prob_matrix[r, c]),
            ))
        return matches

    times = sorted(ids_by_t)
    for t in times:
        source_ids = ids_by_t.get(t, []) #obtains the node ids for that time
        target_ids = ids_by_t.get(t + 1, []) #obtains the node ids for time + 1
        if not source_ids or not target_ids:
            continue
        unmatched_sources = set(source_ids) #turns those node ids into a set
        unmatched_targets = set(target_ids) #turns those target ids into a set
        frame_matches: list[tuple[int, int, float, float, str, float]] = []
        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), 
                                   ("relaxed", MOTION_RELINK_RELAXED_UM)):
            #it goes for each node_id in source_ids only if it is in unmatched_sources
            #So this is a list of nodes, e.g. [0,1,2,3,...]
            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]
            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]
            matches = assign_pass(pass_sources, pass_targets, gate_um)
            for source_id, target_id, raw, motion, prob in matches:
                #if the source and target are already matched by the previous procedure
                if source_id not in unmatched_sources or target_id not in unmatched_targets:
                    continue
                unmatched_sources.remove(source_id) #removes from the set the nodes already assigned
                unmatched_targets.remove(target_id)
                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))
                if pass_name == "tight":
                    stats["motion_relink_tight_edges"] += 1
                else:
                    stats["motion_relink_relaxed_edges"] += 1
        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:
            selected_edges.append({
                "source_id": source_id,
                "target_id": target_id,
                "edge_prob": prob,
                "distance_um": raw,
                "motion_distance_um": motion,
                "motion_relinked": 1,
                "motion_pass": pass_name,
            })
            predecessor_position_um[target_id] = position_um[source_id]
        stats["motion_relink_frames"] += 1

    stats["motion_relink_edges"] = len(selected_edges)
    return selected_edges

def close_single_frame_gaps(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    '''
    Purpose: After the earlier repair steps, some tracks are broken by exactly one missing frame, 
    a cell was detected at frame t but the detector missed it at t+1, and picked it back up at t+2.
    This function finds those "dangling" track ends and start, and stitches them back together across 
    the 1-frame gap, either by reusing a nearby unclaimed detection as the missing frame,
    or by inventing a synthetic node at the midpoint.
    '''
    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:
        return nodes_by_id, edges

    # --- separates from "edges" what links are outgoing (source) and which incoming (targets)
    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    incident = outgoing | incoming #computes the union of the outgoing and incoming sets

    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    isolated_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items(): #remeberd: node is a dict
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id) #that node ends there
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id) #there is no previous node to this one, it starts at t
        if node_id not in incident: #it is an isolated node, no incoming or outgoing node
            isolated_by_t.setdefault(t, []).append(node_id)

    #maximum number of synthetic nodes to add in the skipt time step t
    max_synthetic = min(
        GAP_CLOSE_MAX_ADDED_ABS,
        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC >
        0 else 0,
    )
    next_id = _next_node_id(nodes_by_id) #returns the next node_id relative to the last one
    #for example, if the last node id is 100 (the key) it will return 101
    
    frame_cache: dict[int, np.ndarray] = {}
    used_starts: set[int] = set()
    used_isolated: set[int] = set()
    synthetic_added = 0
    new_edges: list[dict[str, object]] = []

    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1) #temporal gap that will be closed
    #only one time gap is supported, t, t+1 and t+2, there is no link beween t ant t+1 but 
    #there is between t and t+2 (one time step gap)
    stats["gap_close_effective_max_gap"] = effective_gap_max
    
    for gap in range(1, effective_gap_max + 1):#this loop goes only once (1 to 2)
        for t, end_ids in sorted(ends_by_t.items()):#iterates through nodes with not outgoing link
            #t:time, end_ids: node_id

            #take node_id that start: no incoming link but outgoing link
            #but taking it with the gap+1 time step
            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]
            if not end_ids or not start_ids:
                continue
            # === obtains the node_ids coordinates (in voxel coordinates) for end_ids and start_ids ==
            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]
            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]
            #==============================================================
            threshold_um = GAP_CLOSE_UM * (gap + 1)#GAP_CLOSE_UM: 5.9 um by default
            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)
            for i, ep in enumerate(end_points): #goes through the coordinates of the endpoints
                for j, sp in enumerate(start_points): #goes through the coordinates of the startpoints
                    d[i, j] = point_distance_um(ep, sp) #computes the physical euclidian distance
                    #between the end and start points (remember they are separated by time: t+gap+1)
            # ===============================================================
            stats["gap_candidates"] += int((d <= threshold_um).sum()) #counts between all distances
            # "d" that were computed how many of those are within the "threshold_um"
            if not np.isfinite(d).any():#in case there is no numerical value for these distances
                continue

            big = threshold_um * 1000.0 + 1.0 #creates a hugh "big" threshold
            #not it is creating a cost function; where the distances is lower than the
            #threshold it assigns the distance other wise a big penalty "b", "cost (i,j)"
            cost = np.where(d <= threshold_um, d, big)
            #Applies the Hungarian algorithm to this distance-cost function. It finds the pair
            # (i,j) which gives the lowest cost. It means the nodes "end" and "start" separated
            #by t+gap+1 in time with the lowest distance.
            row_ind, col_ind = linear_sum_assignment(cost)
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > threshold_um:
                    continue
                source_id = end_ids[int(r)] #assigns to "source_id" the id in "end_ids"
                target_id = start_ids[int(c)] #similar as above
                #the idea is now to link one node that ended with one that doesnt have a precedent link (start)
                if source_id in outgoing or target_id in used_starts:
                    continue

                #access to the dictionary (t,x,y,z) of the node_id (source_id or target_id)
                source = nodes_by_id[source_id]
                target = nodes_by_id[target_id]
                #=============================================================0
                mid_t = int(source["t"]) + gap #computes the time t+gap. Remember that the node
                #we obtained as "start node" was at t+gap+1
                # Naive node creation: creates one node at coordinates x,y,z which are the 
                # midpoint between start and end node.
                mid_point = (
                    (float(source["z"]) + float(target["z"])) / 2.0,
                    (float(source["y"]) + float(target["y"])) / 2.0,
                    (float(source["x"]) + float(target["x"])) / 2.0,
                )
                # ===================================================================
                
                middle_id: int | None = None
                if GAP_CLOSE_REUSE_EXISTING:
                    #obtains as "candidates" isolated nodes that are at t* = t+gap
                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]
                    if candidates:
                        #computes the distance between the node candidates and the mid point
                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) 
                                     for nid in candidates]
                        #selects the index with the isolated point closer to the mid-point
                        best_idx = int(np.argmin(distances))
                        #if the lowest distance is smaller than GAP... sets this node as the
                        #bridge between end and start node
                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:
                            middle_id = candidates[best_idx]
                            used_isolated.add(middle_id)
                            stats["gap_reused_existing"] += 1
                # === this part creates a synthetic node if there is no isolated node =======
                # === closer enough to end and start nodes that can be assigned as the ======
                # === bridge node ===========================================================
                if middle_id is None:
                    if synthetic_added >= max_synthetic:
                        stats["gap_skipped_node_cap"] += 1
                        continue
                    middle_id = next_id #set the "middle_id" (which is the node bridge) to the 
                    #"next_id" which is a new id for the creation of a synthetic node.
                    next_id += 1 #increments the id node counter by 1
                    #mid_t: time where the gap is presented
                    #frame_cache: empty dictionary
                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, 
                                                              frame_cache, stats)
                    # ========= add the synthetic  node to the list of nodes ===============
                    nodes_by_id[middle_id] = {
                        "node_id": middle_id,
                        "t": mid_t,
                        "z": refined_point[0],
                        "y": refined_point[1],
                        "x": refined_point[2],
                    }
                    synthetic_added += 1
                    stats["gap_inserted_synthetic"] += 1

                middle = nodes_by_id[middle_id]
                # ==== Adds the new links (edges) to the total record of edges ===========
                e1 = {
                    "source_id": source_id,
                    "target_id": middle_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(source, middle),
                    "gap_closed": 1,
                }
                e2 = {
                    "source_id": middle_id,
                    "target_id": target_id,
                    "edge_prob": None,
                    "distance_um": edge_distance_um(middle, target),
                    "gap_closed": 1,
                }
                new_edges.extend([e1, e2])
                outgoing.add(source_id)
                incoming.add(middle_id)
                outgoing.add(middle_id)
                incoming.add(target_id)
                used_starts.add(target_id)
                stats["gap_pairs_selected"] += 1
                stats["gap_added_edges"] += 2

    if new_edges:
        edges = [*edges, *new_edges] #merge edges and new_edges
    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]
    return nodes_by_id, edges


def _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    by_source: dict[int, list[int]] = {}
    for edge in edges:
        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))
    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}


def _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:
    '''
    builds a lookup: for every node that has exactly one incoming edge, map it to its single parent's ID. 
    Makes a dict with key target and value parent
    '''
    by_target: dict[int, list[int]] = {}
    for edge in edges:
        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))
    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}


def recover_strict_gap2(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
    dataset: str | None = None,
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    
    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:
        return nodes_by_id, edges

    # === Splits the nodes into "outgoing" and "incoming" whether the link comes or leaved the node ====
    outgoing = {int(edge["source_id"]) for edge in edges}
    incoming = {int(edge["target_id"]) for edge in edges}
    # ============================================================================
    # == builds relation ships between source/target -> target/source that has only one link ====
    predecessor = _single_predecessor_map(edges)#list(dict) with key: target_id and value parent_id
    successor = _single_successor_map(edges) #same idea as above but for key: parent and value: target
    #it outputs only parents with one single child.
    # ========================================================================
    # = constructs a dictionary with the time as key, dividing those nodes that doesn't have
    # = an outgoing edge (ends_by_t) and those that don't have an incoming edge (starts_by_t)
    ends_by_t: dict[int, list[int]] = {}
    starts_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        t = int(node["t"])
        if node_id not in outgoing:
            ends_by_t.setdefault(t, []).append(node_id)
        if node_id not in incoming:
            starts_by_t.setdefault(t, []).append(node_id)
    # ==================================================================================

    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))
    proposals: list[tuple[float, int, int, int, float]] = []

    def pos_um(node_id: int) -> np.ndarray:
        '''
        Transform the z,y,x voxel coordiantes to physical coordinates
        '''
        node = nodes_by_id[node_id] #this gives a dictionary with t,z,y,x
        return np.array([float(node["z"]), float(node["y"]), float(node["x"])],
                        dtype=np.float64) * np.array(VOXEL_SCALE_UM)

    for t, end_ids in sorted(ends_by_t.items()):#t is time and ends_ids are ids
        start_ids = starts_by_t.get(t + 3, [])# this is the gap, it is taking the starting nodes
        #3 time steps ahead of the ending node
        
        if not end_ids or not start_ids:
            continue
        for end_id in end_ids:# goes through all ids for that time
            end_pos = pos_um(end_id)#obtains the physical coordinates as an array [z,y,x]
            for start_id in start_ids: #goes through the nodes at t+3
                start_pos = pos_um(start_id) #gest the physical position of this node
                dist = float(np.linalg.norm(start_pos - end_pos))#computes the physical distance between
                #start and end nodes
                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:
                    continue #if the distance is too large, it continues to the next node
                step = (start_pos - end_pos) / 3.0
                context_penalty = 0.0
                if GAP2_REQUIRE_CONTEXT:
                    ok_context = False
                    prev_id = predecessor.get(end_id)#this access the key: target and obtains the partent pointing to that target
                    if prev_id is not None: #checks that there was a target with one parent
                        prev_step = end_pos - pos_um(prev_id) #computes the end_pos (at time t)
                        #distance to the parent of the node end_id. in other words, is there a 
                        #parent of the ending node (at t) and what is the distance of this parent to its child
                        prev_norm = float(np.linalg.norm(prev_step))#ditance between end_point and its parent
                        step_norm = float(np.linalg.norm(step))
                        if prev_norm <= 0.01 or step_norm <= 0.01:#these two are almost located at the
                            #same position
                            ok_context = True
                        else:
                            # computes the cosine of the angle between the "motion before 
                            #the gap" vector and the "proposed bridge" vector 
                            #cos θ = (a·b)/(|a||b|: 1->same direction, -1->opposite directions
                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:
                                ok_context = True
                            #adds a penalty if the direccion of both nodes is opposite
                            context_penalty += max(0.0, 0.25 - cos)
                    # ====== Same idea as for the previous block, but for the starting node =======
                    next_id = successor.get(start_id)#now is taking the target of the starting node
                    if next_id is not None:
                        next_step = pos_um(next_id) - start_pos
                        next_norm = float(np.linalg.norm(next_step))
                        step_norm = float(np.linalg.norm(step))
                        if next_norm <= 0.01 or step_norm <= 0.01:
                            ok_context = True
                        else:
                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))
                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:
                                ok_context = True
                            context_penalty += max(0.0, 0.25 - cos)
                    if not ok_context: #if the nodes are too far away they won't be considered
                        continue
                #these are the propossed nodes to be linked. The first element is the cost function
                #which is proportional to the distance between the nodes and the direction
                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))

    # At this point, all ending and starting nodes have been studied and the propossed nodes are
    # stored in "proposals"
    proposals.sort(key=lambda item: item[0]) #sort (lowest) the proposed candidate links by the cost function
    stats["gap2_candidates"] = len(proposals) #metadata
    if not proposals:
        return nodes_by_id, edges #if there are no proposals, the graph is not modified

    selected: list[tuple[float, int, int, int, float]] = []
    used_ends: set[int] = set()
    used_starts: set[int] = set()
    per_frame_count: dict[int, int] = {}
    #=== This "for loop" creates "selected", "used_ends", "used_starts", "per_frame_count" ====
    #== from the "proposals" limiting the amount of elemenst to defined limits and unique ===
    #== node ids so there are no duplicated candidates. So the pair (end_id, start_id) ===
    #== with the lowest cost function is considered only once and not another combination of ==
    #== (end_id, start_id02) ====================
    for proposal in proposals: #goes trhough each propossed link, list.
        if len(selected) >= cap: #checks if the number of considered candidates is larger than the cap
            stats["gap2_skipped_cap"] += 1
            break
        _, end_id, start_id, t, _ = proposal #it ignores the cost and distance of the propossed link
        if end_id in used_ends or start_id in used_starts: #in case these node_ids have already been used
            continue
        #computes the number of nodes that end at t and multply them by the defined frac_cap
        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))
        if per_frame_count.get(t, 0) >= frame_cap: #checks if the number of considered of possible links at time t
            #have reach the maximum allowed value
            continue
        selected.append(proposal)
        used_ends.add(end_id)
        used_starts.add(start_id)
        per_frame_count[t] = per_frame_count.get(t, 0) + 1
    # ===========================================================================================
    if not selected:
        return nodes_by_id, edges
    # 
    next_node_id = _next_node_id(nodes_by_id)#returns the next node_id relative to the last one
    #for example, if the last node id is 100 (the key) it will return 101
    frame_cache: dict[int, np.ndarray] = {} #the same idea as the 1 gap recover, stores intensity data 
    #of the nodes at time t in case this same time step is called again dont read the file two times
    new_edges: list[dict[str, object]] = [] #to store the creation of new edges
    for _, end_id, start_id, t, _ in selected: #not using cost neither distance of the propossed nodes
        source = nodes_by_id[end_id]
        target = nodes_by_id[start_id]
        previous_id = end_id
        inserted_ids: list[int] = []
        # === Here is where the artificial nodes for the gap t +1 and t+2 are created =====
        for k in (1, 2):
            frac = k / 3.0
            mid_t = int(source["t"]) + k
            midpoint = (
                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,
                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,
                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,
            )
            # here the refinement of the naive position is done by using the intensity of surrounding nodes
            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)
            node_id = next_node_id #assignt to "node_id" this artificial created id
            next_node_id += 1 #adss one artificil id to the counter 
            nodes_by_id[node_id] = { #add the artificial node to the list of nodes
                "node_id": node_id,
                "t": mid_t,
                "z": refined_point[0],
                "y": refined_point[1],
                "x": refined_point[2],
            }
            inserted_ids.append(node_id) #keep tracks of the inserted node_id
            current = nodes_by_id[node_id] #assigns to "current" the freshly created node
            new_edges.append({ #creates the link between the source to the new node
                "source_id": previous_id,
                "target_id": node_id,
                "edge_prob": None,
                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),
                "gap2_recovered": 1,
            })
            previous_id = node_id #defines this new node as the previous node
        new_edges.append({ #final link between the last artificial created node and the real "start" node
            "source_id": previous_id,
            "target_id": start_id,
            "edge_prob": None,
            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),
            "gap2_recovered": 1,
        })
        stats["gap2_pairs_selected"] += 1
        stats["gap2_added_nodes"] += len(inserted_ids)
        stats["gap2_added_edges"] += 3

    return nodes_by_id, [*edges, *new_edges]#return the added nodes and links


def add_safe_divisions_postlink(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> list[dict[str, object]]:
    '''
    This function connectes a node without a incoming link at t+1 with a node at t. Always that
    this parent node is within certain distance of the candidate node and that the parent node already
    has a child node at t+1 which is also within a distance with respect to the candidate node.
    '''
    
    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:
        return edges

    out_by_source: dict[int, list[dict[str, object]]] = {}
    incoming: set[int] = set()
    for edge in edges:
        #out_by_source with key:source_id and the list is all times for the same source t,z,y,x,...
        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)
        incoming.add(int(edge["target_id"])) #"incoming" only stores the id

    ids_by_t: dict[int, list[int]] = {}
    for node_id, node in nodes_by_id.items():
        ids_by_t.setdefault(int(node["t"]), []).append(node_id) #stores all nodes id at given time t

    #this creates the pair (source_id, target_id)
    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}
    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))
    added: list[dict[str, object]] = []
    used_targets: set[int] = set()

    for t in sorted(ids_by_t):#moves through time, all nodes at time t
        child_frame_ids = ids_by_t.get(t + 1, []) #gets the nodes at t+1
        if not child_frame_ids:
            continue
        #obtains all node ids at time t for which the have only one edge (they have only one child)
        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]
        #here it is taking the nodes at t+1 that don't have an incoming edge (no parent)
        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in
                         incoming and node_id not in used_targets]
        if not source_ids or not candidate_ids:
            continue

        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))
        proposals: list[tuple[float, int, int, float, float]] = []
        for source_id in source_ids: #iterates through the id of possible parents
            source = nodes_by_id[source_id]#takes all info for that id: t,z,y,x, 
            existing_child_edge = out_by_source[source_id][0] #similar to above, where the node id
            #is taken, but it takes the first element of the list. "out_by_source" contains for a
            #specific node id all edges (links) to oder nodes
            existing_child_id = int(existing_child_edge["target_id"])#takes the target_id
            existing_child = nodes_by_id.get(existing_child_id)#now it takes the t,y,x info for the target
            #which is a child of "source_id"

            #this is checking that the child exists and that the child doesn't have a temporal gap
            if existing_child is None or int(existing_child["t"]) != t + 1:
                continue
            #Computes the physical (multiplied by the voxel scale) euclidian distance (in 3D)
            #between the two connected nodes
            child_dist = edge_distance_um(source, existing_child)
            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:
                continue
            for candidate_id in candidate_ids:#now it moves through all candidates which are identified
                #nodes but witout a parent
                if (source_id, candidate_id) in existing_edges:
                    #safe guard, in theory, candidate_id doesn't have any "source_id"
                    continue
                candidate = nodes_by_id[candidate_id]#obtains the info for "candidate_id"
                parent_dist = edge_distance_um(source, candidate)#obains the distance between the
                #possible parent and the candidate
                if parent_dist > SAFE_DIV_MAX_UM:
                    continue
                sister_dist = edge_distance_um(existing_child, candidate)
                #computes the distance between the confirmed child and the possible child
                if sister_dist > SAFE_DIV_SISTER_MAX_UM:
                    continue
                #creates a "score" where the sister_dist receives a smaller weigth than parent distance
                score = parent_dist + 0.15 * sister_dist
                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))

        stats["safe_division_candidates"] += len(proposals)
        if not proposals:
            continue
        proposals.sort(key=lambda item: item[0])#sort the proposals ascending, lower distance is first
        added_this_frame = 0
        for _, source_id, candidate_id, parent_dist, _ in proposals:
            if len(added) >= global_cap:
                stats["safe_division_skipped_cap"] += 1
                break
            if added_this_frame >= frame_cap:
                break
            #checks that this candidate has not already been used as a child
            if candidate_id in used_targets or candidate_id in incoming:
                continue
            candidate = nodes_by_id[candidate_id]
            added.append({
                "source_id": source_id,
                "target_id": candidate_id,
                "edge_prob": None,
                "distance_um": parent_dist,
                "safe_division": 1,
            })
            used_targets.add(candidate_id)
            added_this_frame += 1

    if added:
        stats["safe_divisions_added"] = len(added)
        return [*edges, *added]
    return edges


def filter_short_track_components(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:
    '''
    len(members) >= OUTPUT_MIN_TRACK_LEN 
    i.e., "is this track long enough to keep, or is it a short spurious fragment to discard?" 
    
    '''
    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:
        return nodes_by_id, edges

    parent = {node_id: node_id for node_id in nodes_by_id}#collects the keys (node_id)

    def find(node_id: int) -> int:
        '''
        Makes sure that parent[node_id] = node_id if this is not the case
        it does a "while" until it is the case.
        '''
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(a: int, b: int) -> None:
        if a not in parent or b not in parent:
            return
        ra = find(a) 
        rb = find(b)
        if ra != rb: #redefines parent, initially it is key:node_id, value:node_id
            #but now it is key:source_id, value:target_id
            parent[ra] = rb
    # ========= Union-Find (also known as Disjoint Set Union, DSU): "which element belongs ===
    #========== to the same group" =============================================
    out_count: dict[int, int] = {}
    for edge in edges:
        #collects source and target ids across the edges
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        #both source and target need to be in parent, i.e. they have to be a node
        union(source_id, target_id)
        #counts how many times source_id appears in edges,i.e. the same source_id how many children has
        out_count[source_id] = out_count.get(source_id, 0) + 1

    components: dict[int, list[int]] = {}
    # components produces the linage, for example if 1→2, 1→3, 2→4, 4→5, 4→6, 7→8, 8→9, 8→10, 11→12
    # components will be: components = {
    # 6:  [1, 2, 3, 4, 5, 6],
    # 10: [7, 8, 9, 10],
    # 12: [11, 12],
    # }
    for node_id in nodes_by_id:
        components.setdefault(find(node_id), []).append(node_id)

    keep: set[int] = set()
    for members in components.values():
        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)
        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):
            keep.update(members)

    if not keep:
        stats["short_track_filter_skipped_all"] += 1
        return nodes_by_id, edges

    removed_nodes = len(nodes_by_id) - len(keep)
    if removed_nodes <= 0:
        return nodes_by_id, edges

    # ====== here is where nodes and links are discarded if they lineage is not long enough ===========
    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}
    kept_edges = [
        edge for edge in edges
        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes
    ]
    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))
    stats["short_track_nodes_removed"] = removed_nodes
    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)
    return kept_nodes, kept_edges


def linefit_smooth_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    edges: list[dict[str, object]],
    stats: dict[str, int],
) -> dict[int, dict[str, object]]:
    """Smooth linear track interiors without changing graph topology."""
    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:
        return nodes_by_id

    predecessor: dict[int, list[int]] = {}
    successor: dict[int, list[int]] = {}
    for edge in edges:
        source_id = int(edge["source_id"])
        target_id = int(edge["target_id"])
        source = nodes_by_id.get(source_id)
        target = nodes_by_id.get(target_id)
        if source is None or target is None:
            continue
        if int(target["t"]) != int(source["t"]) + 1:
            continue
        successor.setdefault(source_id, []).append(target_id)
        predecessor.setdefault(target_id, []).append(source_id)

    original_pos = {
        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)
        for node_id, node in nodes_by_id.items()
    }
    updated_pos: dict[int, np.ndarray] = {}
    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))

    for node_id in sorted(nodes_by_id):
        neighbourhood: list[tuple[int, int]] = [(0, node_id)]

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            prev_ids = predecessor.get(current, [])
            if len(prev_ids) != 1:
                break
            current = prev_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((-step, current))

        current = node_id
        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):
            next_ids = successor.get(current, [])
            if len(next_ids) != 1:
                break
            current = next_ids[0]
            if current not in original_pos:
                break
            neighbourhood.append((step, current))

        if len(neighbourhood) < 3:
            stats["linefit_skipped_nodes"] += 1
            continue

        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)
        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])
        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)
        if not np.isfinite(fitted).all():
            stats["linefit_skipped_nodes"] += 1
            continue
        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted

    for node_id, pos in updated_pos.items():
        nodes_by_id[node_id]["z"] = float(pos[0])
        nodes_by_id[node_id]["y"] = float(pos[1])
        nodes_by_id[node_id]["x"] = float(pos[2])

    stats["linefit_smoothed_nodes"] = len(updated_pos)
    return nodes_by_id


def filter_output_graph(
    nodes_by_id: dict[int, dict[str, object]],
    raw_edges: list[dict[str, object]], #list of dictionaries connecting nodes of each video (.geff)
    dataset: str | None = None,#path to the current .geff file
) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:
    '''
    This function goes trhough each edge (link) and computes the distance between connected edges.
    This computed distance is added to the edge dictionary. Later, it creates a dictionary
    "learned_edge_probs[key]" where the key is the tuple(source_node_id, target_node_id) and the value
    is the probability of that link happening (source->target node).
    '''
    
    stats = {
        "raw_edges": len(raw_edges),
        "dropped_nonconsecutive_edges": 0,
        "dropped_long_edges": 0,
        "dropped_multi_parent_edges": 0,
        "dropped_multi_child_edges": 0,
        "dropped_division_edges": 0,
        "gap_candidates": 0,
        "gap_pairs_selected": 0,
        "gap_reused_existing": 0,
        "gap_inserted_synthetic": 0,
        "gap_added_nodes": 0,
        "gap_added_edges": 0,
        "gap_skipped_node_cap": 0,
        "gap_refined_synthetic": 0,
        "gap_refine_failed": 0,
        "gap_refine_rejected_shift": 0,
        "pruned_isolated_nodes": 0,
        "motion_relink_edges": 0,
        "motion_relink_tight_edges": 0,
        "motion_relink_relaxed_edges": 0,
        "motion_relink_frames": 0,
        "motion_relink_replaced_raw_edges": 0,
        "motion_relink_fallback_raw": 0,
        "motion_relink_skipped_large_frame": 0,
        "gap2_candidates": 0,
        "gap2_pairs_selected": 0,
        "gap2_added_nodes": 0,
        "gap2_added_edges": 0,
        "gap2_skipped_cap": 0,
        "safe_division_candidates": 0,
        "safe_divisions_added": 0,
        "safe_division_skipped_cap": 0,
        "short_track_components_removed": 0,
        "short_track_nodes_removed": 0,
        "short_track_edges_removed": 0,
        "short_track_filter_skipped_all": 0,
        "linefit_smoothed_nodes": 0,
        "linefit_skipped_nodes": 0,
    }
    # ============ This block is intended to modify "stats" using the edge dictionary and to =============
    # ============ create a new dictionary "edge" with the edges that didn't modify the stats ============
    # ============ it means: the node links were consectuive in time and they are inside the ============
    # ============ distance range "OUTPUT_EDGE_MAX_UM" ===================================================
    edges: list[dict[str, object]] = []
    for edge in raw_edges:
        source = nodes_by_id.get(int(edge["source_id"])) #obtains the dictionary for that node id (t,x,y,z)
        target = nodes_by_id.get(int(edge["target_id"]))
        if source is None or target is None:
            continue #goes to the new edge
        # by default OUTPUT_ENFORCE_NEXT_FRAME = True
        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:
            #This means that the next_frame edge continuity is reinforced. When the
            #flag is set to true and the target and source nodes don't have consecutive times
            stats["dropped_nonconsecutive_edges"] += 1
            continue
        #Computes the physical (multiplied by the voxel scale) euclidian distance (in 3D)
        #between the two connected nodes
        distance_um = edge_distance_um(source, target)
        edge["distance_um"] = distance_um #add a new column (or key) to the edge dictionary
        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:
            stats["dropped_long_edges"] += 1
            continue
        edges.append(edge) #only those edges continues in time (t,t+1) and within a distance
    # ================================================================================================
    # ===== Creates a new dictionary "learned_edge_probs" with key the tuple source_node, target_node =
    # ===== and obtains the probability that that link happens ========================================
    if OUTPUT_MOTION_RELINK:
        learned_edge_probs: dict[tuple[int, int], float] = {}
        for edge in edges: #moves through the "edges" created in the block above
            prob = edge.get("edge_prob")
            if prob is None:
                continue
            try:
                prob = float(prob)
            except (TypeError, ValueError):
                continue
            if np.isfinite(prob):
                key = (int(edge["source_id"]), int(edge["target_id"]))
                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)
        # =============================================================================================
        
        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs)
        if motion_edges:
            stats["motion_relink_replaced_raw_edges"] = len(edges)
            edges = motion_edges
        else:
            stats["motion_relink_fallback_raw"] = 1
    #========  enforces the "single parent" rule: each node may have at most ==========
    #======== one incoming edge, i.e. no cell can be claimed as the "child" ==========
    #======== of two different parent cells. ==========
    if OUTPUT_SINGLE_PARENT_REPAIR and edges: 
        best_by_target: dict[int, dict[str, object]] = {}
        for edge in edges: #Goes through each edge
            target_id = int(edge["target_id"]) #obtains the target id of that specific edge
            prev = best_by_target.get(target_id) #for the first time, it returns None
            # "edge_sort_key" gives the prob and -distance of that edge for source-target nodes
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_target[target_id] = edge #assigns the edge to that "target_id". edge is a dict
                
        kept_ids = {id(edge) for edge in best_by_target.values()}# "id()" returns an integer that uniquely identifies 
                                                            #an object for the duration of its lifetime
        #checkig how many edges are not in kept_ids, how many were descarded
        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids] #make a list of the edges dict
        #that survived
    # ===========================================================================================

    #======== This is similar as the previous block, but this is enforcing that one parent cell ==
    #=== can have only 1 child, this is unrealistic in practice as through mitosis ===========
    #== one cell can divide into 2 cells, the default value is off: OUTPUT_SINGLE_CHILD_REPAIR=False
    if OUTPUT_SINGLE_CHILD_REPAIR and edges:
        best_by_source: dict[int, dict[str, object]] = {}
        for edge in edges:
            source_id = int(edge["source_id"])
            prev = best_by_source.get(source_id)
            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):
                best_by_source[source_id] = edge
        kept_ids = {id(edge) for edge in best_by_source.values()}
        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)
        edges = [edge for edge in edges if id(edge) in kept_ids]
    #=================================================================================================
    #nodes_by_id:this is the raw input, edges is the outptu from the Hungrian algorithm,
    #stats is the long dict, dataset is the path to the dataset (.geff file)
    # "close_single_frame_gaps" creates a node or uses an isolaed node to bridge two nodes
    # that are not connected by one of them ends (no outgoing link) and the other starts
    # (no incoming link) and they are separate in time by gap+1 and within certain distance-
    # the bridge node is located at time = gap
    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset)
    
    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)
    
    edges = add_safe_divisions_postlink(nodes_by_id, edges, stats)
    
    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:
        by_source: dict[int, list[dict[str, object]]] = {}
        for edge in edges:
            by_source.setdefault(int(edge["source_id"]), []).append(edge)

        filtered: list[dict[str, object]] = []
        for source_id, source_edges in by_source.items():#"source_edges" is the list containing
            #all edged for that source
            if len(source_edges) <= 1:#that source has only one or less edge (link)
                filtered.extend(source_edges)
                continue

            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)#sorts in reverse order
            #the function "edge_sort_key" returns a tuple (prob,-dist) so the larger prob and
            #subsequently (in case of a tie) the smaller distance rank first
            source = nodes_by_id[source_id]
            #takes the top 2 childs for that source
            top1 = ranked[0]
            top2 = ranked[1]
            d1 = float(top1["distance_um"])
            d2 = float(top2["distance_um"])
            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], 
                                      nodes_by_id[int(top2["target_id"])])
            valid_division = (
                max(d1, d2) <= DIV_PARENT_MAX_UM
                and sister <= DIV_SISTER_MAX_UM
                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1
                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1
            )
            if valid_division:
                filtered.extend([top1, top2])
                stats["dropped_division_edges"] += max(0, len(ranked) - 2)
            elif DIV_DROP_TO_SINGLE_IF_BAD:
                filtered.append(top1)
                stats["dropped_division_edges"] += len(ranked) - 1
            else:
                filtered.extend(ranked)
        edges = filtered

    if OUTPUT_PRUNE_ISOLATED:
        #constructs a set with "source_id" and another for "target_id" (without duplicates, sets
        #dont accept duplicates) and computes the union ("|") between both sets
        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}
        if incident:
            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}
            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)
            nodes_by_id = kept_nodes
            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and 
                     int(edge["target_id"]) in nodes_by_id]
    
    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)
    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)

    return nodes_by_id, edges, stats

# %%%%%%%% search for all predictions inside folder "predictions" %%%%%%%%%%%%%%%%%%%%%%%%%%
geffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))#searches inside .../predictions/ for all *.geff files
print(f"Found {len(geffs)} prediction graphs")

# %%%%% As each test video should produce one .geff file, it checks that %%%%%%%%%%%%%%%%%
# %%%%% the number of predictions matched with the number of test files %%%%%%%%%%%%%%%
if len(geffs) != len(test_stems):
    found = {path.stem for path in geffs}
    missing = sorted(set(test_stems) - found)
    raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

stats_rows: list[dict[str, object]] = []
seen_datasets: set[str] = set()
row_id = 0
total_nodes = 0
total_edges = 0

with SUBMISSION_PATH.open("w", newline="") as f: #opens the "submission.csv" file in write mode
    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS) #creates a writer object that writes rows to the open file f
    # in the specified column order ("CSV_COLUMNS")
    writer.writeheader() #immediately writes that header row, as the first line of "submission.csv"

    for geff_path in geffs: #goes through each one of the .geff outputs
        dataset = geff_path.stem #take the path to that file
        seen_datasets.add(dataset) #add the path to the set (now it has been seen)
        graph = graph_from_geff(geff_path) #reads the graph

        nodes_by_id: dict[int, dict[str, object]] = {} #creates a new dictionary for each .geff file (video)
        # ========= Construcst a dictionary with all the attributes of each node ============
        for row in graph.node_attrs().iter_rows(named=True):
            node_id = int(row["node_id"])
            nodes_by_id[node_id] = {
                "node_id": node_id,
                "t": int(row["t"]),
                "z": float(row["z"]),
                "y": float(row["y"]),
                "x": float(row["x"]),
            }
        # =================================================================================
        # ========= Construcst a list with all the attributes of each edge ============
        raw_edges: list[dict[str, object]] = []
        for row in graph.edge_attrs().iter_rows(named=True):
            edge_prob = row.get("edge_prob") if hasattr(row, "get") else None
            raw_edges.append({
                "source_id": int(row["source_id"]),
                "target_id": int(row["target_id"]),
                "edge_prob": None if edge_prob is None else float(edge_prob),
            })

        raw_node_count = len(nodes_by_id)
        # Main function, it performs all the post-processing improvements
        nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset)
        # === ACA VOY
        if not nodes_by_id:
            raise AssertionError(f"{dataset}: post-processing removed every node")

        for node_id in sorted(nodes_by_id):#goes through each node_id (int) key of the dictionaries
            node = nodes_by_id[node_id]#access to the dictionary, which contains all data
            #writes each noce as a new row
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "node",
                "node_id": int(node["node_id"]),
                "t": int(node["t"]),
                "z": int(round(float(node["z"]))),
                "y": int(round(float(node["y"]))),
                "x": int(round(float(node["x"]))),
                "source_id": -1,
                "target_id": -1,
            })
            row_id += 1
        # ===================================================================================
        division_sources: dict[int, int] = {}
        for edge in edges: #goes trhough each edge row
            source_id = int(edge["source_id"])
            target_id = int(edge["target_id"])
            if source_id not in nodes_by_id or target_id not in nodes_by_id:
                #it means that this link (edge->source) doesn't correspond to recognized nodes
                raise AssertionError(f"{dataset}: dangling edge after filtering")
            writer.writerow({
                "id": row_id,
                "dataset": dataset,
                "row_type": "edge",
                "node_id": -1,
                "t": -1,
                "z": -1,
                "y": -1,
                "x": -1,
                "source_id": source_id,
                "target_id": target_id,
            })
            row_id += 1
            #keeps track of how many times a source id has a target (child)
            division_sources[source_id] = division_sources.get(source_id, 0) + 1
        # =====================================================================================
        node_count = len(nodes_by_id)
        edge_count = len(edges)
        # === These 2 variable keep track of the total amount of nodes and edges across all videos
        total_nodes += node_count
        total_edges += edge_count
        # ================================================================================0
        stats_rows.append({
            "dataset": dataset,
            "raw_nodes": raw_node_count,
            "nodes": node_count,
            "raw_edges": filter_stats["raw_edges"],
            "edges": edge_count,
            "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),
            "edge_to_node_ratio": edge_count / max(node_count, 1),
            "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(raw_node_count, 1),
            **filter_stats,
        })

expected_datasets = set(test_stems)
missing_datasets = sorted(expected_datasets - seen_datasets)
extra_datasets = sorted(seen_datasets - expected_datasets)
if missing_datasets or extra_datasets:
    raise AssertionError({"missing": missing_datasets[:10], "extra": extra_datasets[:10]})
assert row_id == total_nodes + total_edges, "Internal row counter mismatch"
assert total_nodes > 0, "No node rows produced"

header = SUBMISSION_PATH.open().readline().strip().split(",") #read the headre of the created file
assert header == CSV_COLUMNS, f"Bad CSV header: {header}"#checks the file and the input header
                                    #matchSUBMISSION_PATH
#==== Creates "stats" dataset =============
stats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)
stats["predict_minutes_total"] = predict_seconds / 60.0
stats["experiment_tag"] = EXPERIMENT_TAG
stats.to_csv(RUN_STATS_PATH, index=False)
#==========================================

print(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")
print(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")
print(f"Wrote {RUN_STATS_PATH}")
display(stats.describe(include="all"))
display(pd.read_csv(SUBMISSION_PATH, nrows=8))


# Cell description

it takes the raw model-predicted tracking graphs and turns them into a validated submission.csv, applying the "deterministic graph repair" described earlier in the notebook's markdown. Here's the breakdown:

Imports (936–939)

tracksdata — loads/manipulates the predicted cell-tracking graphs (nodes = cell detections, edges = frame-to-frame links).
blosc2 / zarr — read the compressed raw microscopy volumes (needed later for sub-voxel refinement).
scipy.optimize.linear_sum_assignment — solves the Hungarian/assignment problem for the motion-based relinking step.
Helper utilities (946–1057)

graph_from_geff — loads one .geff graph file (one video's full predicted lineage).
edge_distance_um / point_distance_um / _position_um — convert voxel coordinates to physical micron distances using VOXEL_SCALE_UM = (1.625, 0.40625, 0.40625) (anisotropic z/y/x voxel size).
edge_sort_key — ranks edges by learned probability, then by shortest distance (used to pick "best" edge among competitors).
read_test_frame — lazily loads and caches one 3D frame from the raw zarr/blosc2 volume (used only for refining synthetic node positions).
Repair passes (each gated by a config flag from the earlier config cell, each updates a running stats dict for diagnostics):

motion_relink_edges (1059–1177) — Rather than trusting the raw predicted edges directly, this rebuilds frame-to-frame links using a Hungarian assignment: for each source node it predicts an expected next position (source + velocity), then solves for the lowest-cost one-to-one matching to next-frame targets, combining motion-distance, raw distance, and the model's learned edge probability into one cost. It runs a "tight" gate first, then a looser "relaxed" gate for whatever's left unmatched.

Single-parent/single-child repair (1779–1799 in filter_output_graph) — enforces at most one incoming edge per node (and optionally one outgoing), keeping the best-ranked edge and dropping the rest — cleans up spurious merges.

close_single_frame_gaps / recover_strict_gap2 (1179–1475) — bridges tracks across 1 missing frame or 2 missing frames respectively, either reusing an existing nearby detection or inserting a synthetic node at the interpolated midpoint (refined against the real image intensity via refine_synthetic_midpoint), subject to strict distance/count caps so it can't run away and hallucinate too many links.

add_safe_divisions_postlink (1476–1562) — conservatively adds cell-division edges (one parent → two children) only when geometry constraints (parent-to-child distance, sister-cell distance) are tightly satisfied.

Division geometry filter (1805–1837) — for any node with >1 outgoing edge, checks if it's plausibly a real division (children close together and in the next frame); if not, keeps only the single best edge.

filter_short_track_components (1563–1622) — removes track fragments below a minimum length, optionally preserving ones connected to a division.

linefit_smooth_output_graph (1623–1696) — smooths node coordinates along each track by fitting a local line, reducing pixel jitter.

Main loop (1853–1959)

Finds every predicted .geff file, asserts one exists per expected test dataset.
For each dataset: loads nodes/edges into plain dicts, runs the entire filter_output_graph pipeline above, then streams the resulting nodes and edges as rows directly to submission.csv (avoiding holding everything in memory).
Tracks per-dataset stats (node/edge counts, how many gap-fills/relinks/divisions happened, etc.) into run_stats.csv for later inspection.
Runs several sanity assertions: no dataset missing/extra, row-count consistency, correct CSV header, at least one node written.
Finally prints/display a summary (stats.describe()) and a preview of the submission file.
In short: cell 1 (the config cell you asked about earlier) defined how aggressive each repair should be; this cell is where those knobs actually get applied to convert the raw learned graph into the final, geometry-sane submission.